In [ ]:
import pandas as pd

In [ ]:
df= pd.read_csv("/content/Stanza_AI_Intent_Classification_Final_700.csv")
print("Dataset shape: ", df.shape)
print("\n dataset columns:")
print(df.columns.tolist())

Dataset shape:  (700, 7)

 dataset columns:
['record_id', 'agent_label', 'intent_name', 'query', 'entities', 'language', 'split']


In [ ]:
print("Split Distribution :")
print(df["split"].value_counts())

print("\n Agent distributions:")
print(df["agent_label"].value_counts())

print("\n First 5 Records:")
display(df.head())

Split Distribution :
split
train         560
validation     70
test           70
Name: count, dtype: int64

 Agent distributions:
agent_label
reservations               100
frontdesk                  100
HousekeepingAgent          100
FnBAgent                   100
ConciergeAgent             100
RevenuePricingAgent        100
ReportingAnalyticsAgent    100
Name: count, dtype: int64

 First 5 Records:


,record_id,agent_label,intent_name,query,entities,language,split
0,27,reservations,reservation_upsell,Can I add a early check-in package to my reser...,"{""addon"": ""early check-in package""}",en,train
1,37,reservations,reservation_rate_question,What's the price of a family room for a week?,"{""room_type"": ""family room"", ""nights"": ""a week""}",en,train
2,49,reservations,reservation_modify,Can I change my booking from December 24th to ...,"{""original_date"": ""December 24th"", ""new_date"":...",en,train
3,56,reservations,reservation_rate_question,Can you tell me the price for five nights in a...,"{""nights"": ""five nights"", ""room_type"": ""standa...",en,train
4,60,reservations,reservation_availability,Do you have any garden-view rooms free this th...,"{""room_type"": ""garden-view room"", ""date"": ""thi...",en,train


# Checking intent distribution

In [ ]:
print("Total Unique Intent", df["intent_name"].nunique())

print("\n Intent distribution:")
display(
    df.groupby(["agent_label", "intent_name"])
    .size()
    .reset_index(name= "count")
    .sort_values(["agent_label", "intent_name"])
)

Total Unique Intent 59

 Intent distribution:


,agent_label,intent_name,count
0,ConciergeAgent,billing_query,10
1,ConciergeAgent,booking_inquiry,13
2,ConciergeAgent,complaint_submission,10
3,ConciergeAgent,facility_information_query,12
4,ConciergeAgent,general_assistance,9
5,ConciergeAgent,guest_feedback,9
6,ConciergeAgent,hotel_information_query,12
7,ConciergeAgent,local_recommendation_request,12
8,ConciergeAgent,room_request,13
9,FnBAgent,beverage_order,5


In [ ]:
print("Duplicate queries :", df["query"].duplicated().sum())
print("\n missing values:")
print(df.isnull().sum())

Duplicate queries : 0

 missing values:
record_id      0
agent_label    0
intent_name    0
query          0
entities       0
language       0
split          0
dtype: int64


Checking duplicate value existance

In [ ]:
duplicates= df[df["query"].duplicated(keep=False)].sort_values("query")
print("Total duplicate rows: ", len(duplicates))

display(
    duplicates[
        ["record_id", "agent_label", "intent_name", "query", "split"]
    ].head(50)
)

Total duplicate rows:  0


,record_id,agent_label,intent_name,query,split


In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets scikit-learn

In [ ]:
!pip install -q -U --force-reinstall pyarrow

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import pandas as pd
import json

def build_prompt(query):
  return f"""you are an intent classifier for the hotel AI System called Stanza AI. Given a guest or staff query identify the responsible agent and specific intent.

Query: {query}
Answer (JSON only):"""

def build_target(agent_label, intent_name):
  return json.dumps({"agent_label": agent_label, "intent_name": intent_name})

df["prompt"] = df["query"].apply(build_prompt)
df["target"] = df.apply(lambda r: build_target(r["agent_label"], r["intent_name"]), axis=1)

df["full_text"]= df["prompt"] + "" + df["target"]

print(df[["query", "prompt", "target"]].iloc[0]["prompt"])
print()
print("Target:", df.iloc[0], ["target"])

you are an intent classifier for the hotel AI System called Stanza AI. Given a guest or staff query identify the responsible agent and specific intent.

Query: Can I add a early check-in package to my reservation?
Answer (JSON only):

Target: record_id                                                     27
agent_label                                         reservations
intent_name                                   reservation_upsell
query          Can I add a early check-in package to my reser...
entities                     {"addon": "early check-in package"}
language                                                      en
split                                                      train
prompt         you are an intent classifier for the hotel AI ...
target         {"agent_label": "reservations", "intent_name":...
full_text      you are an intent classifier for the hotel AI ...
Name: 0, dtype: object ['target']


Splitting data into train/test/validation

In [ ]:
from datasets import Dataset
train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df = df[df["split"]== "test"].reset_index(drop=True)
val_df = df[df["split"]== "validation"].reset_index(drop=True)

print("Train size:", len(train_df))
print("Test Size:", len(test_df))
print("Validation size:", len(val_df))

# now converting dataset in the form that huggingface understands

train_dataset = Dataset.from_pandas(train_df[["full_text"]])
test_dataset = Dataset.from_pandas(test_df[["prompt", "target", "intent_name", "agent_label"]])
val_dataset = Dataset.from_pandas(val_df[["prompt", "target", "intent_name", "agent_label"]])

Train size: 560
Test Size: 70
Validation size: 70


In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

## Quantization -- compressing size so that it can fit for t4 (16gb)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded:", MODEL_NAME)
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-7B-Instruct
GPU memory used (GB): 5.56


LoRa Configuration

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model= prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha= 32,
    target_modules= ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM",
)

model= get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


#Actual Fine Tunning Dataset step using LoRa

In [ ]:
from trl import SFTTrainer, SFTConfig

training_par = SFTConfig(
    output_dir = "./qwen-stanza-finetune",
    num_train_epochs= 3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps = 10,
    save_strategy= "epoch",
    eval_strategy ="epoch",
    bf16=True,
    max_length=512,
    dataset_text_field= "full_text",
    report_to = "none",

)

trainer = SFTTrainer(
    model= model,
    args= training_par,
    train_dataset= train_dataset,
    eval_dataset= Dataset.from_pandas(val_df[["full_text"]]),

)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/70 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/70 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.432350,0.418365,0.409337,36527.000000,0.901436
2,0.348225,0.354524,0.357591,73054.000000,0.912998
3,0.287129,0.338086,0.301274,109581.000000,0.916654


TrainOutput(global_step=210, training_loss=0.5172302961349488, metrics={'train_runtime': 3230.7108, 'train_samples_per_second': 0.52, 'train_steps_per_second': 0.065, 'total_flos': 4788741954877440.0, 'train_loss': 0.5172302961349488, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Evaluating model on Test Dataset

In [ ]:
import re
import json as jsonlib
from tqdm import tqdm

model.eval()

def generate_answer(prompt, max_new_tokens=60):
  inputs = tokenizer(prompt, return_tensors= "pt").to(model.device)
  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens= max_new_tokens,
        do_sample=False,
        pad_token_id= tokenizer.pad_token_id,
    )
    generated= tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

def extract_json(text):
  match= re.search(r'\{.*?}', text, re.DOTALL)
  if match:
    try:
      return jsonlib.loads(match.group())
    except:
      return None

results = []
for row in tqdm(test_dataset):
  raw_output = generate_answer(row["prompt"])
  parsed = extract_json(raw_output)
  pred_agent = parsed.get("agent_label") if parsed else None
  pred_intent = parsed.get("intent_name") if parsed else None

  results.append({
      "query_prompt": row["prompt"],
      "true_agent": row["agent_label"],
      "true_intent": row["intent_name"],
      "pred_agent": pred_agent,
      "pred_intent": pred_intent,
      "raw_output": raw_output,
  })

  results_df = pd.DataFrame(results)
  results_df.head(10)

100%|██████████| 70/70 [08:05<00:00,  6.94s/it]


In [ ]:
import shutil
shutil.copytree(
    "./qwen-stanza-finetune/checkpoint-210",
    "/content/drive/MyDrive/stanza_ai_qwen_checkpoint",
    dirs_exist_ok=True
)
print("Model backed up to Google Drive!")

Model backed up to Google Drive!


In [ ]:
results_df.to_csv("/content/drive/MyDrive/qwen_evaluation_results.csv", index=False)
print("Evaluation results saved to Drive!")

Evaluation results saved to Drive!


#Final Accuracy

In [ ]:
results_df["fully_correct"] = (results_df["pred_agent"] == results_df["true_agent"]) & (results_df["pred_intent"] == results_df["true_intent"])
results_df["agent_correct"] = results_df["pred_agent"] == results_df["true_agent"]
results_df["intent_correct"] = results_df["pred_intent"] == results_df["true_intent"]

overall_accuracy = results_df["fully_correct"].mean() * 100
agent_accuracy = results_df["agent_correct"].mean() * 100
intent_accuracy = results_df["intent_correct"].mean() * 100

print(f"Overall Accuracy (agent+intent combined): {overall_accuracy:.2f}%")
print(f"Agent-only Accuracy: {agent_accuracy:.2f}%")
print(f"Intent-only Accuracy: {intent_accuracy:.2f}%")

Overall Accuracy (agent+intent combined): 88.57%
Agent-only Accuracy: 97.14%
Intent-only Accuracy: 88.57%


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("✅ Drive Connected!")

Mounted at /content/drive
✅ Drive Connected!


In [ ]:
import os
import shutil
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("✅ All libraries imported!")

✅ All libraries imported!


In [ ]:
import shutil
import os

# Source aur destination
source = '/content/drive/MyDrive/stanza_ai_qwen_checkpoint'
destination = '/content/drive/MyDrive/stanza_ai_qwen_final_model'

# Check if source exists
if os.path.exists(source):
    print("✅ Source checkpoint found!")

    # If destination already exists, delete it
    if os.path.exists(destination):
        shutil.rmtree(destination)
        print("🔄 Old final model removed")

    # Copy checkpoint to final model
    shutil.copytree(source, destination)
    print("✅ Checkpoint copied to final model!")

    # Verify
    files = os.listdir(destination)
    print(f"\n✅ Model saved with {len(files)} files:")
    for f in files[:5]:  # First 5 files
        print(f"   └─ {f}")

else:
    print("❌ Source checkpoint not found!")

✅ Source checkpoint found!
✅ Checkpoint copied to final model!

✅ Model saved with 11 files:
   └─ training_args.bin
   └─ trainer_state.json
   └─ scheduler.pt
   └─ tokenizer.json
   └─ rng_state.pth
